# Document Question Answering System (RAG)


## Overview

This project implements a Retrieval-Augmented Generation (RAG) system that answers questions based on custom documents.

Instead of relying only on a language model's internal knowledge, the system retrieves relevant information from documents and then generates answers grounded in that information. This improves factual accuracy and allows question answering over private or domain-specific data.

## System Architecture

The pipeline consists of the following stages:

1. **Document Ingestion** — documents such as PDFs or text files are loaded and converted into raw text.
2. **Text Chunking** — the text is split into smaller chunks to improve retrieval accuracy.
3. **Embedding Creation** — each chunk is converted into a vector representation capturing its semantic meaning.
4. **Vector Database** — embeddings are stored in a vector database (FAISS) for efficient similarity search.
5. **Query Processing** — the user's question is converted into an embedding.
6. **Context Retrieval** — the system retrieves the most relevant chunks from the database.
7. **Answer Generation** — a language model generates an answer using the retrieved context.


## Setup

In [1]:
%pip install -q sentence-transformers faiss-cpu transformers sentencepiece protobuf PyPDF2 accelerate

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import textwrap
import numpy as np
import faiss
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer

import warnings
warnings.filterwarnings("ignore")

## Stage 1 — Document Ingestion

**Data:** input sources include PDF documents, text files, and notes/articles — typically unstructured and possibly containing domain-specific knowledge.

In [3]:
DOC_PATH = "sample_notes.txt"   

def load_text(path: str) -> str:
    """Load raw text from a .pdf or .txt file (Document Ingestion stage)."""
    if path.lower().endswith(".pdf"):
        reader = PdfReader(path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        return text
    else:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()

if not os.path.exists(DOC_PATH):
    sample_text = """
    Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval
    with text generation. Instead of relying only on the knowledge stored in a language model's
    parameters, a RAG system first retrieves relevant chunks of text from an external knowledge
    source (such as a document or database) and then passes that context to the language model
    to generate a grounded, up-to-date answer.

    A typical RAG pipeline has three main stages: Indexing, Retrieval, and Generation.
    In the Indexing stage, documents are split into smaller chunks and converted into
    vector embeddings using an embedding model such as Sentence-BERT. These embeddings are
    stored in a vector database like FAISS, Pinecone, or Chroma.

    In the Retrieval stage, when a user asks a question, the question itself is embedded into
    the same vector space, and a similarity search (usually cosine similarity or L2 distance)
    is performed to find the top-k most relevant chunks from the index.

    In the Generation stage, the retrieved chunks are combined with the original question inside
    a prompt template, and this combined prompt is passed to a large language model (LLM) such
    as GPT, LLaMA, or Flan-T5, which produces the final answer grounded in the retrieved context.

    RAG systems are useful because they reduce hallucination, allow models to answer questions
    about private or recent data that was not part of their training set, and are cheaper to
    update than retraining a model, since only the document store needs to change.

    Common challenges in RAG systems include choosing the right chunk size, handling documents
    where the answer spans multiple chunks, and ranking retrieved chunks so that the most
    relevant ones appear first.
    """
    with open(DOC_PATH, "w", encoding="utf-8") as f:
        f.write(sample_text)
    print(f"No document found — created a sample file at '{DOC_PATH}' for demonstration.")

raw_text = load_text(DOC_PATH)
print(f"Loaded document with {len(raw_text)} characters.")
print(raw_text[:500])

Loaded document with 1803 characters.

    Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval
    with text generation. Instead of relying only on the knowledge stored in a language model's
    parameters, a RAG system first retrieves relevant chunks of text from an external knowledge
    source (such as a document or database) and then passes that context to the language model
    to generate a grounded, up-to-date answer.

    A typical RAG pipeline has three main stages: Indexing, Retrieval, a


## Stage 2 — Text Chunking

Large documents are split into smaller chunks so that each chunk fits within the embedding
model's context window and retrieval can be more precise.


In [4]:
def _is_section_header(line: str) -> bool:
    """A short, all-caps line is treated as a resume/report section header (e.g. SKILLS)."""
    line = line.strip()
    if not (2 <= len(line) <= 40):
        return False
    letters = [c for c in line if c.isalpha()]
    if not letters:
        return False
    return all(c.isupper() for c in letters) and line.upper() == line


def chunk_text(text: str, chunk_size: int = 100, overlap: int = 20) -> list:
    """Split text into chunks (Text Chunking stage).

    Uses section-aware chunking when the document has detectable headers (resumes, reports),
    otherwise falls back to fixed-size overlapping word chunks (plain notes, articles, prose).
    """
    lines = [l for l in text.splitlines() if l.strip()]
    header_indices = [i for i, l in enumerate(lines) if _is_section_header(l)]

    def word_window_chunks(flat_text: str) -> list:
        words = re.sub(r"\s+", " ", flat_text).strip().split(" ")
        result, start = [], 0
        while start < len(words):
            end = start + chunk_size
            result.append(" ".join(words[start:end]))
            start += chunk_size - overlap
        return result

    if len(header_indices) < 2:
        return word_window_chunks(text)

    sections = []
    for idx, h_idx in enumerate(header_indices):
        end_idx = header_indices[idx + 1] if idx + 1 < len(header_indices) else len(lines)
        sections.append(" ".join(lines[h_idx:end_idx]))

    if header_indices[0] > 0:
        sections.insert(0, " ".join(lines[:header_indices[0]]))

    chunks = []
    for section in sections:
        words = section.split(" ")
        if len(words) <= chunk_size:
            chunks.append(section)
            continue
        header = words[0]
        start = 0
        while start < len(words):
            end = start + chunk_size
            sub = " ".join(words[start:end])
            if not sub.startswith(header):
                sub = f"{header} (cont.) {sub}"
            chunks.append(sub)
            start += chunk_size - overlap
    return chunks


chunks = chunk_text(raw_text, chunk_size=100, overlap=20)
print(f"Document split into {len(chunks)} chunks.\n")
for i, c in enumerate(chunks):
    print(f"--- Chunk {i} ---")
    print(textwrap.fill(c, 100))
    print()

Document split into 4 chunks.

--- Chunk 0 ---
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text
generation. Instead of relying only on the knowledge stored in a language model's parameters, a RAG
system first retrieves relevant chunks of text from an external knowledge source (such as a document
or database) and then passes that context to the language model to generate a grounded, up-to-date
answer. A typical RAG pipeline has three main stages: Indexing, Retrieval, and Generation. In the
Indexing stage, documents are split into smaller chunks and converted into vector embeddings using
an embedding model such as Sentence-BERT. These embeddings are stored in

--- Chunk 1 ---
into smaller chunks and converted into vector embeddings using an embedding model such as Sentence-
BERT. These embeddings are stored in a vector database like FAISS, Pinecone, or Chroma. In the
Retrieval stage, when a user asks a question, the question itself is embe

## Stage 3 & 4 — Embedding Creation & Vector Database

Each chunk is converted into a dense vector using the `all-MiniLM-L6-v2` sentence-transformer
model (fast and lightweight — good for a beginner-friendly RAG pipeline). The vectors are then
stored in a **FAISS** vector database for efficient similarity search.

In [5]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embed_model.encode(chunks, convert_to_numpy=True, show_progress_bar=True)
embedding_dim = chunk_embeddings.shape[1]

faiss.normalize_L2(chunk_embeddings)

index = faiss.IndexFlatIP(embedding_dim)   
index.add(chunk_embeddings)

print(f"FAISS index built with {index.ntotal} vectors of dimension {embedding_dim}.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index built with 4 vectors of dimension 384.


## Stage 5 & 6 — Query Processing & Context Retrieval

The user's question is converted into an embedding (Query Processing), and the system searches
the FAISS index for the `top_k` most similar chunks (Context Retrieval).

In [6]:
def retrieve(query: str, top_k: int = 3):
    top_k = min(top_k, index.ntotal) 
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, top_k)
    results = [
        (chunks[idx], float(score))
        for idx, score in zip(indices[0], scores[0])
        if idx != -1   
    ]
    return results

# Quick sanity check
test_results = retrieve("What is RAG used for?", top_k=3)
for rank, (chunk, score) in enumerate(test_results, start=1):
    print(f"Rank {rank} | similarity={score:.3f}")
    print(textwrap.fill(chunk, 100))
    print()

Rank 1 | similarity=0.483
in RAG systems include choosing the right chunk size, handling documents where the answer spans
multiple chunks, and ranking retrieved chunks so that the most relevant ones appear first.

Rank 2 | similarity=0.441
the original question inside a prompt template, and this combined prompt is passed to a large
language model (LLM) such as GPT, LLaMA, or Flan-T5, which produces the final answer grounded in the
retrieved context. RAG systems are useful because they reduce hallucination, allow models to answer
questions about private or recent data that was not part of their training set, and are cheaper to
update than retraining a model, since only the document store needs to change. Common challenges in
RAG systems include choosing the right chunk size, handling documents where the answer spans
multiple chunks, and ranking retrieved

Rank 3 | similarity=0.436
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text
generatio

## Stage 7 — Answer Generation


In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GEN_MODEL_NAME = "google/flan-t5-base"
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME)

def build_prompt(query: str, context_chunks: list) -> str:
    context = "\n\n".join(context_chunks)
    prompt = (
        "You are a precise assistant. Answer the question using ONLY the facts in the context "
        "below. Give a short, direct answer (a list of items, a name, or a number) -- do not "
        "copy section headings or unrelated text. "
        "If the answer is not contained in the context, say 'I don't know based on the document.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return prompt

def generate_answer(query: str, context_chunks: list, max_new_tokens: int = 200) -> str:
    prompt = build_prompt(query, context_chunks)
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = gen_model.generate(**inputs, max_new_tokens=max_new_tokens)
    answer = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return answer.strip()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


## Full Workflow

Putting it all together: **load & preprocess documents -> split into chunks -> convert to
embeddings -> store in vector database -> accept user query -> retrieve relevant chunks ->
generate answer using retrieved context.**


In [8]:
def rag_answer(query: str, top_k: int = 3, verbose: bool = True):
    retrieved = retrieve(query, top_k=top_k)
    context_chunks = [c for c, _ in retrieved]
    answer = generate_answer(query, context_chunks)

    if verbose:
        print(f"Q: {query}\n")
        print(f"A: {answer}\n")
        print("Sources used:")
        for i, (chunk, score) in enumerate(retrieved, start=1):
            print(f"  [{i}] (similarity={score:.3f}) {textwrap.fill(chunk, 90)}")
        print("\n" + "-"*100 + "\n")

    return answer, retrieved

## Example Flow

**User Question:** "What is the main idea of the document?"

**System Process:** retrieves relevant sections -> provides them as context -> generates a
concise answer.

In [9]:
questions = [
    "What is the main idea of the document?",
    "What are the three main stages of a RAG pipeline?",
    "Why are RAG systems useful?",
    "What is the capital of France?",   # out-of-context question to test grounding
]

for q in questions:
    rag_answer(q, top_k=3)

Q: What is the main idea of the document?

A: RAG systems include choosing the right chunk size, handling documents where the answer spans multiple chunks, and ranking retrieved chunks so that the most relevant ones appear first.

Sources used:
  [1] (similarity=0.251) in RAG systems include choosing the right chunk size, handling documents where the answer
spans multiple chunks, and ranking retrieved chunks so that the most relevant ones appear
first.
  [2] (similarity=0.247) Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval
with text generation. Instead of relying only on the knowledge stored in a language
model's parameters, a RAG system first retrieves relevant chunks of text from an external
knowledge source (such as a document or database) and then passes that context to the
language model to generate a grounded, up-to-date answer. A typical RAG pipeline has three
main stages: Indexing, Retrieval, and Generation. In the Indexing stage, docume

## Validation Log

A consolidated log of retrieval + generation performance across the sample questions above --
useful for demonstrating that retrieval is actually finding relevant, high-similarity chunks
(and for spotting cases where it isn't, such as the deliberately out-of-context question).

In [10]:
import pandas as pd

def build_validation_log(query_list, top_k=3):
    rows = []
    for q in query_list:
        answer, retrieved = rag_answer(q, top_k=top_k, verbose=False)
        top_score = retrieved[0][1] if retrieved else float("nan")
        top_chunk_preview = textwrap.shorten(retrieved[0][0], width=80) if retrieved else "N/A"
        rows.append({
            "question": q,
            "top_similarity": round(top_score, 3),
            "top_chunk_preview": top_chunk_preview,
            "generated_answer": answer,
        })
    return pd.DataFrame(rows)

validation_log = build_validation_log(questions, top_k=3)
validation_log

,question,top_similarity,top_chunk_preview,generated_answer
0,What is the main idea of the document?,0.251,in RAG systems include choosing the right chun...,RAG systems include choosing the right chunk s...
1,What are the three main stages of a RAG pipeline?,0.434,in RAG systems include choosing the right chun...,"Indexing, Retrieval, and Generation."
2,Why are RAG systems useful?,0.539,in RAG systems include choosing the right chun...,they reduce hallucination
3,What is the capital of France?,0.076,into smaller chunks and converted into vector ...,I don't know based on the document.


## Stage 8 — System Optimization Experiments


### Experiment 1 — Adjusting Chunk Borders

Re-chunk the same document with a smaller `chunk_size` and compare how many chunks are produced
and how retrieval for a sample query changes.

In [11]:
chunks_small = chunk_text(raw_text, chunk_size=50, overlap=10)
chunks_large = chunk_text(raw_text, chunk_size=150, overlap=30)

print(f"Default  (size=100, overlap=20): {len(chunks)} chunks")
print(f"Smaller  (size=50,  overlap=10): {len(chunks_small)} chunks")
print(f"Larger   (size=150, overlap=30): {len(chunks_large)} chunks")
print()
print("Smaller chunks -> more precise retrieval but less surrounding context per chunk.")
print("Larger chunks  -> more context per chunk but retrieval precision can drop as topics mix.")

Default  (size=100, overlap=20): 4 chunks
Smaller  (size=50,  overlap=10): 7 chunks
Larger   (size=150, overlap=30): 3 chunks

Smaller chunks -> more precise retrieval but less surrounding context per chunk.
Larger chunks  -> more context per chunk but retrieval precision can drop as topics mix.


### Experiment 2 — Hybrid Search (Keyword + Vector)

Pure dense-vector retrieval can occasionally rank a chunk highly just because of a single
overlapping keyword. Hybrid search combines the FAISS similarity score with a simple 
keyword-overlap score, so chunks that match both semantically *and* lexically are ranked higher.

In [12]:
def keyword_overlap_score(query: str, chunk: str) -> float:
    """Simple keyword-overlap score: fraction of query words that literally appear in the chunk."""
    q_words = set(re.findall(r"\w+", query.lower()))
    c_words = set(re.findall(r"\w+", chunk.lower()))
    if not q_words:
        return 0.0
    return len(q_words & c_words) / len(q_words)

def hybrid_retrieve(query: str, top_k: int = 3, alpha: float = 0.7):
    """alpha weights dense similarity vs keyword overlap (alpha=1.0 -> pure vector search)."""
    # over-fetch a larger candidate pool from FAISS, then re-rank with the hybrid score
    candidate_k = min(index.ntotal, max(top_k * 3, top_k))
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, candidate_k)

    candidates = []
    for idx, vec_score in zip(indices[0], scores[0]):
        if idx == -1:
            continue
        chunk = chunks[idx]
        kw_score = keyword_overlap_score(query, chunk)
        hybrid_score = alpha * float(vec_score) + (1 - alpha) * kw_score
        candidates.append((chunk, hybrid_score, float(vec_score), kw_score))

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:top_k]

test_query = "What was Shivang's role in the Smart India Hackathon project?"
print("--- Pure vector search (top 3) ---")
for chunk, score in retrieve(test_query, top_k=3):
    print(f"vec_score={score:.3f} | {textwrap.shorten(chunk, 90)}")

print("\n--- Hybrid search (top 3) ---")
for chunk, hybrid_score, vec_score, kw_score in hybrid_retrieve(test_query, top_k=3):
    print(f"hybrid={hybrid_score:.3f} (vec={vec_score:.3f}, kw={kw_score:.2f}) | {textwrap.shorten(chunk, 90)}")

--- Pure vector search (top 3) ---
vec_score=0.068 | Retrieval-Augmented Generation (RAG) is a technique that combines information [...]
vec_score=0.057 | the original question inside a prompt template, and this combined prompt is passed [...]
vec_score=0.047 | into smaller chunks and converted into vector embeddings using an embedding model [...]

--- Hybrid search (top 3) ---
hybrid=0.129 (vec=0.068, kw=0.27) | Retrieval-Augmented Generation (RAG) is a technique that combines information [...]
hybrid=0.122 (vec=0.057, kw=0.27) | the original question inside a prompt template, and this combined prompt is passed [...]
hybrid=0.087 (vec=0.047, kw=0.18) | into smaller chunks and converted into vector embeddings using an embedding model [...]


## System Metrics Report

A summary of the configuration used in this pipeline -- useful for reproducibility and for
evaluators to see the chunking profile, embedding setup, vector store, and generation model at a
glance.

In [13]:
chunk_lengths = [len(c.split()) for c in chunks]

system_metrics = {
    "document_source": DOC_PATH,
    "document_characters": len(raw_text),
    "chunking_method": "section-aware (fallback: fixed word-window)",
    "chunk_size_words": 100,
    "chunk_overlap_words": 20,
    "num_chunks": len(chunks),
    "avg_chunk_length_words": round(sum(chunk_lengths) / len(chunk_lengths), 1),
    "min_chunk_length_words": min(chunk_lengths),
    "max_chunk_length_words": max(chunk_lengths),
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dimension": embedding_dim,
    "vector_store": "FAISS IndexFlatIP (cosine similarity via L2-normalized inner product)",
    "generation_model": GEN_MODEL_NAME,
    "default_top_k": 3,
}

pd.DataFrame(system_metrics.items(), columns=["metric", "value"])

,metric,value
0,document_source,sample_notes.txt
1,document_characters,1803
2,chunking_method,section-aware (fallback: fixed word-window)
3,chunk_size_words,100
4,chunk_overlap_words,20
5,num_chunks,4
6,avg_chunk_length_words,82.2
7,min_chunk_length_words,29
8,max_chunk_length_words,100
9,embedding_model,sentence-transformers/all-MiniLM-L6-v2


## Key Learnings

- How RAG systems combine retrieval and generation to produce grounded, factual answers
- The importance of retrieval quality in improving overall answer accuracy
- Practical experience working with embeddings and vector databases (FAISS)
- Handling unstructured text data (PDFs, notes) end-to-end
- Designing a scalable AI pipeline made of clearly separated stages (ingestion, chunking, embedding, retrieval, generation)
- Hybrid search (keyword + vector) can correct cases where pure dense retrieval is misled by
  coincidental keyword overlap between unrelated sections

## Conclusion

This project demonstrates how to build a system that can understand user queries, retrieve
relevant information from custom documents, and generate accurate, grounded answers using an LLM.

